# 3.  Star Catalogues

In this tutorial we show how you can create a custom star catalogue. Later we show how to create a synthetic star catalogue. We note that realistic star catalogues can be generated as part of the PLATOnium toolkit (which another notebook is dedicated for).

### Setup notebook

In [ ]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# To interact with the plot use
%matplotlib notebook

### Imports

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from mpl_interactions import ipyplot as iplt
from prettytable import PrettyTable

# PlatoSim
import platosim.referenceFrames as rf
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.matplotlibrc import setup_notebook
setup_notebook()

---
## 3.1 - Generate a *custom* star catalog
---

As a first example we show how to create a small custom stellar catalogue prior to running PlatoSim. As always we first define the inputs and outputs:

In [ ]:
# Inputs
inputDir    = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles"
inputFile   = inputDir + "/inputfile.yaml"

# Outputs
outputDir      = os.getcwd()
outputFileName = "output_example1"

# Set up a Simulation object
sim = Simulation(outputFileName, inputFile, outputDir=outputDir)

You can also cutomize the orientation (pointing and rotation) of the spacecraft before defining your stellar catalogue:

In [ ]:
sim["ObservingParameters/RApointing"]   = 180.0 
sim["ObservingParameters/DecPointing"]  = -70.0
sim["Platform/SolarPanelOrientation"]   = 0.0

We then set a subfield around the stars being large enough to contain all of them ($50\times50$ pixels):

In [ ]:
sim["SubField/ZeroPointRow"]    = 3000
sim["SubField/ZeroPointColumn"] = 3000
sim["SubField/NumColumns"]      = 50
sim["SubField/NumRows"]         = 50

Specify the pixel coordinates (of the CCD, not of the subfield) of your stars, and create the star catalog file: an ascii file will be written with the columns: `ra`, `dec`, and `mag`. We will use the above defined CCD location to set the rows and columns of the stars:

In [ ]:
# Define catalogue
row = np.array([ 7.0, 40.0, 20.0]) + sim["SubField/ZeroPointRow"]
col = np.array([30.0, 45.0,  5.0]) + sim["SubField/ZeroPointColumn"]
mag = np.array([11.0, 10.0, 12.0])
ID  = [0, 1, 2]

# Automatic catalogue file creation
starcatFile = outputDir + "/starcat_example1.txt"
sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, ID, starcatFile)
row, col

We need to tell PlatoSim to use this newly created catalogue:

In [ ]:
# Make sure the simulation object uses this star catalog
sim["ObservingParameters/StarCatalogFile"] = starcatFile

Finally let's run the simulation and show the stellar pixel map:

In [ ]:
simfile = sim.run(removeOutputFile=True)

In [ ]:
fig, ax = simfile.showImage(clipPercentile=1, imgScale="clip", fontSize=20, 
                            colorMap="cubehelix", colorBar=True, showGrid=True);

---
## 3.2 - Generate a *synthetic* star catalog
---

With the above definitions we can start simulting a realistic synthetic stellar catalogue. For this purpose we have created a `Distribution` class which, among others, cn be used to generate synthetic stellar catalogues. Simple load it with:

In [ ]:
from platosim.distribution import Distribution
filename = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles/starcatalog.txt"
dist = Distribution(filename)

Let's show a little (pretty) table of class functions:

In [ ]:
t = PrettyTable()
t.add_column("Distribution utilities", dir(Distribution)[27:])
t

First we can use `getStarCatalog()` to generate a random catalogue:

In [ ]:
# Star catalogue
row, col, mag, ID = dist.getStarCatalog(numStars=50, minMag=8, maxMag=13, 
                                        subfieldNumRows=sim["SubField/NumRows"],
                                        subfieldNumCols=sim["SubField/NumColumns"], 
                                        subfieldZeroRow=sim["SubField/ZeroPointRow"],
                                        subfieldZeroCol=sim["SubField/ZeroPointColumn"],
                                        plot=True)

In [ ]:
# Automatic catalogue file creation
starcatFile = outputDir + "/starcat_example2.txt"
sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, ID, starcatFile)

In [ ]:
# Make sure the simulation object uses this star catalog
sim["ObservingParameters/StarCatalogFile"] = starcatFile

In [ ]:
simfile = sim.run(removeOutputFile=True)

In [ ]:
fig, ax = simfile.showImage(imageNr=False, clipPercentile=1, imgScale="log",  
                            figsize=(8,7), fontSize=20, useTitle=False,
                            showStarPositions=False, showStarIDs=False,
                            colorMap="cubehelix", colorBar=True, showGrid=True);

---
## 3.3 - Multi-subfield *synthetic* catalogue creation
---
## Under development!

In [ ]:
# # Change these parameters after use
# numStars       = 10
# subfieldDim    = 50
# numSubfields   = 5
# numExposures   = 1
# ccdCode        = 4
# outputFileName = outputDir  + "/output_example3_"
# filename       = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles/starcatalog.txt"

In the following we show how to create a stellar catalogue, place the stars on different part of the CCD detectors, and simulate each subfield independently over a loop: 

In [ ]:
# # Sub-field related configuration parameters
# # (1) "lowerLeft": in the lower left corner of the CCD (i.e. close to the readout register,
# #     small PSF)
# # (2) "lowerRight": in the lower right corner of the CCD (i.e. close to the readout register,
# #     large PSF)
# # (3) "upperLeft": in the upper left corner of the CCD (i.e. far from the readout register,
# #     large PSF)
# # (4) "middle": in the centre of the CCD (sub-field overlaps with both detector halves)
# # (5) "upperRight": in the upper right corner of the CCD (i.e. far from the readout register,
# #      large PSF)
# subFieldNames = ["output_example2_lowerLeft", "output_example2_lowerRight", 
#                  "output_example2_upperLeft", "output_example2_middle", 
#                  "output_example2_upperRight"]

# # Set subfield dimentions
# ccdDimensions         = sim["CCDPositions/NumColumns"][0]
# subFieldCenterRows    = [subfieldDim/2, subfieldDim/2, ccdDimensions - subfieldDim/2, 
#                          ccdDimensions/2, ccdDimensions - subfieldDim/2]
# subFieldCenterColumns = [subfieldDim/2,   ccdDimensions - subfieldDim/2, subfieldDim/2, 
#                          ccdDimensions/2, ccdDimensions - subfieldDim/2]

In [ ]:
# # Run each subfield over a lopp

# for i in range(numSubfields):
    
#     # Initialise PlatoSim
#     sim = Simulation(subFieldNames[i], outputDir=outputDir)
    
#     # Obs params   
#     sim["ObservingParameters/NumExposures"] = numExposures
      
#     # CCD params
#     sim["CCD/Position"] = str(ccdCode)
    
#     # (x, y) = (column, row) -> (RA, Dec) [radians]
#     subFieldCenterRa, subFieldCenterDec = rf.pixelToSkyCoordinates(sim, ccdCode, 
#                                                                    subFieldCenterColumns[i], 
#                                                                    subFieldCenterRows[i])
    
#     # Try to set the subfield around the pixel coordinates 
#     sim.setSubfieldAroundPixelCoordinates(ccdCode, subFieldCenterColumns[i], subFieldCenterRows[i], 
#                                           subfieldDim, subfieldDim)

#     # Generate star catalogue
#     starcatFileName = outputFileName +  subFieldNames[i] + ".txt"
    
#     ra = []
#     dec = []
    
#     for starIndex in range(numStars):
        
#         # Generate stellar catalogue
#         row, col, mag, ID = dist.getStarCatalog(numStars, minMag=8, maxMag=13, 
#                                                 subfieldNumRows=subfieldDim,
#                                                 subfieldNumCols=subfieldDim, 
#                                                 subfieldZeroRow=sim["SubField/ZeroPointRow"],
#                                                 subfieldZeroCol=sim["SubField/ZeroPointColumn"],
#                                                 plot=True)
        
        
#         #randomRow, randomColumn = getRandomPosition(subFieldDimensions, subFieldDimensions, subFieldCenterRows[simulationIndex], subFieldCenterColumns[simulationIndex])     # Random position in the sub-field (uniform distribution)
# #         randomRow    = rowsCatalog[starIndex]    + subFieldCenterRows[i]    - subfieldDim / 2
# #         randomColumn = columnsCatalog[starIndex] + subFieldCenterColumns[i] - subfieldDim / 2
        
#         # (x, y) -> (RA, Dec) [radians]
#         randomRa, randomDec = rf.pixelToSkyCoordinates(sim, ccdCode, randomColumn, randomRow)
#         ra.append(randomRa)
#         dec.append(randomDec)
    
#     # Convert to numpy array
#     ra  = np.array(ra)
#     dec = np.array(dec)
    
#     # RA should be in range [-PI, PI] and convert to radians
#     ra[ra > np.pi] = ra[ra > np.pi] - 2*np.pi
#     ra  = np.rad2deg(ra)
#     dec = np.rad2deg(dec)
# #     print(ra, dec, mag)
    
#     # Store the catalogue
#     np.savetxt(starcatFileName, np.transpose([ra, dec, mag]), 
#                fmt=['%11.6f', '%11.6f', '%8.4f'])
#     sim["ObservingParameters/StarCatalogFile"] = starcatFileName
    
#     # Write YAML file 
#     configurationFileName = outputFileName + subFieldNames[i] + ".yaml"
#     sim.writeYamlConfigurationFile(configurationFileName)
    
#     # Run each simulation                      
#     simfile = sim.run(removeOutputFile=True)

In [ ]:
# fig = simfile.showImage(0, clipPercentile=0.5, imgScale="clip",  
#                         figsize=(8,8), fontSize=20, useTitle=False,
#                         showStarPositions=False, showStarIDs=False,
#                         colorMap="cubehelix", colorBar=True, showGrid=True) 